In [1]:
import pandas as pd
import numpy as np
import re ##re means Regular Expressions.It is used for text cleaning and pattern matching.

In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
#Converts text into numerical features based on word importance.
from scipy.sparse import hstack
#Combines text features and metadata features into a single input for the model.

In [3]:
train=pd.read_excel("/content/train.xlsx")
test=pd.read_excel("/content/test.xlsx")

In [4]:
print("training shape:",train.shape)
print("testing shape:",test.shape)

training shape: (1200, 13)
testing shape: (120, 11)


In [5]:
train.head()

,id,journal_text,ambience_type,duration_min,sleep_hours,energy_level,stress_level,time_of_day,previous_day_mood,face_emotion_hint,reflection_quality,emotional_state,intensity
0,1,The ocean ambience helped me stop drifting and...,ocean,12,6.5,4,2,afternoon,mixed,calm_face,clear,focused,3
1,2,"I tried to relax during the forest ambience, y...",forest,35,6.0,2,4,evening,calm,tired_face,vague,restless,3
2,3,The forest session slowed my thoughts and I fe...,forest,3,NaN,2,1,night,overwhelmed,happy_face,clear,calm,3
3,4,"the mountain ambience was pleasant, though i c...",mountain,25,7.0,4,4,night,focused,calm_face,vague,neutral,1
4,5,"The rain session gave me a pause, but the pres...",rain,25,5.0,3,5,afternoon,NaN,tense_face,clear,overwhelmed,5


##### Telling your model what type of data each column is.

#####  Basically,grouping columns into:

##### Text

##### Numbers

#####Categories

In [6]:
text_col="journal_text"

numeric_col=["duration_min",
             "sleep_hours",
             "energy_level",
             "stress_level",
             ]

cat_col=["ambience_type",
         "time_of_day",
         "previous_day_mood",
         "face_emotion_hint",
         "reflection_quality",
         ]


In [7]:
test.head()

,id,journal_text,ambience_type,duration_min,sleep_hours,energy_level,stress_level,time_of_day,previous_day_mood,face_emotion_hint,reflection_quality
0,10001,woke up feeling more organized mentally. i was...,cafe,4,8.5,3,1,night,mixed,happy_face,vague
1,10002,started off distracted most of the time. this ...,mountain,4,8.5,1,2,afternoon,mixed,happy_face,clear
2,10003,kinda calm ...,cafe,15,8.5,2,5,evening,calm,happy_face,vague
3,10004,after the session i felt able to think straigh...,ocean,7,7.0,2,3,morning,overwhelmed,none,clear
4,10005,lowkey felt pretty grounded. i had to restart ...,ocean,20,8.5,1,5,afternoon,calm,tired_face,vague


In [8]:
for col in numeric_col:
    median_val = train[col].median()

    train[col] = train[col].fillna(median_val)
    test[col] = test[col].fillna(median_val)
for col in cat_col:

    train[col] = train[col].fillna("unknown")
    test[col] = test[col].fillna("unknown")

In [9]:
def clean_text(text):
  text=text.lower()
  text=re.sub(r'[^a-zA-Z ]', '', text)
  return text

train["clean_text"] = train[text_col].apply(clean_text)
test["clean_text"] = test[text_col].apply(clean_text)

In [10]:
vectorizer = TfidfVectorizer(
    max_features=3000,
    stop_words="english"
)
all_text = pd.concat([train["clean_text"], test["clean_text"]])#So the model learns all possible words
vectorizer.fit(all_text)#modellearns simple words
train_text = vectorizer.transform(train["clean_text"])#Each sentence becomes a numeric vector
test_text = vectorizer.transform(test["clean_text"])#Same transformation for test data

We combine train and test metadata to ensure consistent encoding, apply one-hot encoding using get_dummies, then split back and convert to numeric format for model compatibility.

In [11]:
combined_metadata=pd.concat([train[numeric_col+cat_col],
                             test[numeric_col+cat_col]])##Metadata =numbers (sleep, stress)categories (morning, forest, happy, etc.)

In [12]:
combined_metadata = pd.get_dummies(combined_metadata)#Machine learning cannot understand words like:morning, forest, happySo we convert them into numbers.

In [13]:
train_meta = combined_metadata.iloc[:len(train)]
test_meta = combined_metadata.iloc[len(train):]##You combined train + test earlierNow you are separating them again.

In [14]:
train_meta = train_meta.astype(float)
test_meta = test_meta.astype(float)##This ensures:all values are numbers no text/object type

In [15]:
X_train = hstack([train_text, train_meta])
X_test = hstack([test_text, test_meta])

In [16]:

y_state = train["emotional_state"]
y_intensity = train["intensity"]

In [17]:
emotion_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

emotion_model.fit(X_train, y_state)

intensity_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

intensity_model.fit(X_train, y_intensity)

print("Models trained successfully")

Models trained successfully


In [18]:

pred_state = emotion_model.predict(X_test)
pred_intensity = intensity_model.predict(X_test)

In [19]:
probs = emotion_model.predict_proba(X_test)

confidence = probs.max(axis=1)

In [20]:
uncertain_flag = (confidence < 0.5).astype(int)

In [21]:

def decision_engine(state, intensity, stress, energy, time):

    if stress >= 4 and intensity >= 3:
        return "box_breathing", "now"

    if state == "tired" and energy <= 2:
        return "rest", "within_15_min"

    if state == "anxious":
        return "grounding", "now"

    if state == "sad":
        return "journaling", "later_today"

    if state == "focused" and energy >= 4:
        return "deep_work", "now"

    if time == "evening":
        return "light_planning", "tonight"

    return "pause", "within_15_min"

In [22]:
def supportive_message(state):

    if state == "anxious":
        return "You seem a bit anxious. Try a short breathing exercise."

    if state == "tired":
        return "You seem tired. A short rest may help recharge."

    if state == "sad":
        return "Writing your thoughts down may help clear your mind."

    if state == "focused":
        return "Great focus. This could be a good moment for deep work."

    return "Take a short pause and reset."

In [23]:
actions = []
times = []
messages = []

for i in range(len(test)):

    state = pred_state[i]
    intensity = pred_intensity[i]

    stress = test.iloc[i]["stress_level"]
    energy = test.iloc[i]["energy_level"]
    time = test.iloc[i]["time_of_day"]

    action, when = decision_engine(
        state,
        intensity,
        stress,
        energy,
        time
    )

    msg = supportive_message(state)

    actions.append(action)
    times.append(when)
    messages.append(msg)


In [24]:
predictions = pd.DataFrame()

predictions["id"] = test["id"]
predictions["predicted_state"] = pred_state
predictions["predicted_intensity"] = pred_intensity
predictions["confidence"] = confidence
predictions["uncertain_flag"] = uncertain_flag
predictions["what_to_do"] = actions
predictions["when_to_do"] = times
predictions["message"] = messages

predictions.to_csv("predictions.csv", index=False)

print("predictions.csv generated successfully")


predictions.csv generated successfully


In [25]:
importances = emotion_model.feature_importances_

print("Top 20 Feature Importance:")
print(importances[:20])

print("Pipeline Completed Successfully")

Top 20 Feature Importance:
[9.02497313e-03 1.45627525e-05 8.15910018e-04 9.71154404e-04
 1.39358802e-03 1.28414541e-04 2.73371038e-03 2.92663040e-03
 5.06948140e-04 5.63388668e-05 4.68309360e-03 2.59040144e-04
 4.21476483e-03 1.58978964e-04 5.98016736e-03 0.00000000e+00
 2.16626281e-04 7.38055150e-03 3.45273075e-03 3.81796872e-04]
Pipeline Completed Successfully


In [26]:
# ==========================================
# 18 USER INPUT (INTERACTIVE MODE)
# ==========================================

print("\n====== USER INPUT MODE ======\n")

# --- Take input ---
user_text = input("Enter your thoughts: ")

sleep = float(input("Sleep hours: "))
energy = int(input("Energy level (1-5): "))
stress = int(input("Stress level (1-5): "))
time_of_day = input("Time of day (morning/evening/night): ")

# --- Clean text ---
user_clean = clean_text(user_text)

# --- TF-IDF transform ---
user_text_vec = vectorizer.transform([user_clean])

# --- Create metadata ---
user_meta_dict = {
    "duration_min": 10,
    "sleep_hours": sleep,
    "energy_level": energy,
    "stress_level": stress,
    "ambience_type": "unknown",
    "time_of_day": time_of_day,
    "previous_day_mood": "unknown",
    "face_emotion_hint": "unknown",
    "reflection_quality": "medium"
}

user_meta = pd.DataFrame([user_meta_dict])

# --- One-hot encoding ---
user_meta = pd.get_dummies(user_meta)

# --- Match training columns ---
user_meta = user_meta.reindex(columns=train_meta.columns, fill_value=0)

# --- Convert to float ---
user_meta = user_meta.astype(float)

# --- Combine features ---
user_X = hstack([user_text_vec, user_meta])

# --- Predict ---
user_state = emotion_model.predict(user_X)[0]
user_intensity = int(intensity_model.predict(user_X)[0])

# --- Confidence ---
user_conf = emotion_model.predict_proba(user_X).max()

# --- Uncertainty ---
user_uncertain = int(user_conf < 0.5)

# --- Decision ---
action, when = decision_engine(
    user_state,
    user_intensity,
    stress,
    energy,
    time_of_day
)

# --- Message ---
msg = supportive_message(user_state)

# --- Output ---
print("\n====== RESULT ======")
print("Emotion:", user_state)
print("Intensity:", user_intensity)
print("Confidence:", round(user_conf, 2))
print("Uncertain Flag:", user_uncertain)
print("What to do:", action)
print("When to do:", when)
print("Message:", msg)


====== USER INPUT MODE ======

Enter your thoughts: happy
Sleep hours: 8
Energy level (1-5): 4
Stress level (1-5): 1
Time of day (morning/evening/night): evening

====== RESULT ======
Emotion: calm
Intensity: 3
Confidence: 0.24
Uncertain Flag: 1
What to do: light_planning
When to do: tonight
Message: Take a short pause and reset.


In [27]:
import pickle

pickle.dump(emotion_model, open("emotion_model.pkl", "wb"))
pickle.dump(intensity_model, open("intensity_model.pkl", "wb"))
pickle.dump(vectorizer, open("vectorizer.pkl", "wb"))
pickle.dump(train_meta.columns, open("train_meta_cols.pkl", "wb"))